# N7 — CFO Decision

## Decision question

What exactly are you asking the CFO to approve, what evidence supports it, and
what would make you change the recommendation?


In [ ]:
from pathlib import Path
import json
import sys

# Find the public package locally. A fresh Colab runtime downloads the same
# participant-safe assets from the repository.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    for source_candidate in (candidate / 'src', candidate / 'CFOPackV002' / 'src'):
        if (source_candidate / 'workshop_bootstrap.py').exists():
            sys.path.insert(0, str(source_candidate))
            break

try:
    from workshop_bootstrap import bootstrap
except ImportError:
    from urllib.request import urlopen
    bootstrap_url = (
        'https://raw.githubusercontent.com/VinayaSharada/'
        'KateelLearningDemosToStudents/cfopack-v002-v2.1.0-beta.1/CFOPackV002/src/workshop_bootstrap.py'
    )
    namespace = {}
    exec(compile(urlopen(bootstrap_url).read(), bootstrap_url, 'exec'), namespace)
    bootstrap = namespace['bootstrap']

ROOT, OUTPUT_DIR = bootstrap()
from cfopack_v002 import (
    analyze_fx,
    default_decisions,
    load_inputs,
    load_manifest,
    reveal_team_shock,
    run_pipeline,
)
import workshop_visuals as viz
import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:
    # Keep the notebooks runnable from a minimal local Python environment as
    # well as Colab/Jupyter. Rich notebook rendering remains the default.
    def Markdown(value):
        return value

    def display(value):
        print(value)

manifest = load_manifest(ROOT / 'config' / 'scenario_manifest.json')
decision_file = OUTPUT_DIR / 'N0_team_decisions.json'
if decision_file.exists():
    DECISIONS = json.loads(decision_file.read_text(encoding='utf-8'))
else:
    DECISIONS = default_decisions(manifest)


In [ ]:
data = load_inputs(ROOT / 'data' / 'synthetic')
viz.data_snapshot(data, OUTPUT_DIR, 'N7')


In [ ]:
summary = run_pipeline(ROOT, OUTPUT_DIR, DECISIONS)
print(f"Scenario {summary['scenario_version']} calculated for {DECISIONS['team_name']} (model cache: {'hit' if summary['model_cache_hit'] else 'rebuilt'})")


## Review the generated decision paper


In [ ]:
paper = (OUTPUT_DIR / 'N7_cfo_decision_paper.md').read_text(encoding='utf-8')
display(Markdown(paper))
evidence = pd.read_csv(OUTPUT_DIR / 'N7_decision_evidence.csv')
display(evidence)
contractual = pd.read_csv(OUTPUT_DIR / 'N2_contractual_forecast.csv')
realistic = pd.read_csv(OUTPUT_DIR / 'N4_realistic_forecast.csv')
selected = pd.read_csv(OUTPUT_DIR / 'N5_selected_action_forecast.csv')
viz.executive_summary(
    contractual, realistic, selected, manifest['minimum_liquidity'], OUTPUT_DIR
)


## CFO challenge preparation

Prepare concise answers to these questions:

1. Why should I trust the model more than a simple rule?
2. Why is this the lowest-cost defensible action package?
3. What happens if collections achieves only half the expected acceleration?
4. Why is the facility draw neither too small nor unnecessarily large?
5. Which approval or policy exception is still unresolved?

Revise the paper before presenting. The generated document is a traceable draft,
not an automatic approval recommendation.


In [ ]:
CFO_DECISION = 'approve'  # approve, revise, reject
approval_record = {
    'status': CFO_DECISION,
    'team': DECISIONS['team_name'],
    'scenario': DECISIONS['scenario_variant'],
    'evidence_file': 'N7_decision_evidence.csv',
}
(OUTPUT_DIR / 'N7_cfo_approval.json').write_text(
    json.dumps(approval_record, indent=2), encoding='utf-8'
)
print(f"CFO decision recorded: {CFO_DECISION.upper()}")


### Before moving on

Record your interpretation in the participant workbook. Do not copy a chart
without also recording the assumption and decision it supports.
